[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/01-working-with-llms/micro-assignment/solution/solution.ipynb)

# Micro-assignment 4.1: reference solution

Each cell prints its result and notes the expected output.

## Setup

```
pip install ollama groq google-genai python-dotenv streamlit
```

### Given helpers and echo provider

In [ ]:
# Given: a cost model and a token estimator. Real providers report exact token
# counts; llm.chat returns only text, so here we ESTIMATE tokens from the text
# (about 1.3 per word) just to exercise the cost and logging path.
def estimate_tokens(text):
    return max(1, round(len(text.split()) * 1.3))

PRICES = {"local": (0.0, 0.0), "hosted": (0.05, 0.08), "frontier": (2.50, 10.0)}
def call_cost(n_in, n_out, tier):
    p_in, p_out = PRICES[tier]
    return n_in / 1e6 * p_in + n_out / 1e6 * p_out

import llm   # the shared real-provider wrapper (gemini/groq/ollama via .env)

### 1. build_messages

In [ ]:
def build_messages(system, user):
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]

msgs = build_messages("You are a terse film buff.", "Give a one-sentence premise for the movie Inception.")
print(msgs)
# -> [{'role': 'system', ...}, {'role': 'user', ...}]

### 2. Cost at 900 in / 300 out

In [ ]:
for tier in PRICES:
    print(f"{tier:>9}: ${call_cost(900, 300, tier):.6f}")
# -> local $0.000000, hosted $0.000069, frontier $0.005250

### 3. Provider switch

In [ ]:
# Problem 3: a provider switch that makes a REAL call and returns text + tokens.
# llm.chat dispatches to the real provider (default ollama) and raises ValueError
# for an unknown provider, so we do not need our own check.
def chat(messages, provider="ollama"):
    text = llm.chat(messages, provider=provider)          # REAL model call
    return {"text": text,
            "in_tokens": estimate_tokens(" ".join(m["content"] for m in messages)),
            "out_tokens": estimate_tokens(text)}

print(chat(msgs)["text"])            # real reply (needs ollama running, or set PROVIDER)
try:
    chat(msgs, provider="nope")
except ValueError as e:
    print("raised:", e)              # unknown provider -> ValueError from llm.chat

### 4. Cost logging

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("llm")
TIER_OF = {"ollama": "local", "groq": "hosted", "gemini": "hosted"}

def chat_logged(messages, provider="ollama"):
    t0 = time.perf_counter()
    out = chat(messages, provider=provider)               # real call
    dt = time.perf_counter() - t0
    cost = call_cost(out["in_tokens"], out["out_tokens"], TIER_OF[provider])
    log.info("provider=%s in=%d out=%d cost=$%.6f latency=%.3fs",
             provider, out["in_tokens"], out["out_tokens"], cost, dt)
    return out

_ = chat_logged(msgs, "ollama")
# -> one INFO line with the real provider, estimated tokens, cost, and real latency

### 5. A/B two tiers (reasoning)

In [ ]:
# Problem 5: make one real call, then price the SAME token counts at two tiers.
out = chat_logged(msgs, "ollama")
for tier in ("hosted", "frontier"):
    print(f"{tier:>9}: cost=${call_cost(out['in_tokens'], out['out_tokens'], tier):.6f}")

# Choice: ship a simple FAQ reply on the hosted tier. It is far cheaper than
# frontier and plenty capable for short, factual answers; reserve frontier for
# tasks where quality is the bottleneck. (Exact numbers vary with the real reply.)